# 임베딩

In [11]:
import json
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "raw_data"
REPO_ROOT = PROJECT_DIR.parent
ENV_FILE = REPO_ROOT / ".env"

MOVIE_CSV = DATA_DIR / "movie.csv"
WEBTOON_CSV = DATA_DIR / "webtoon.csv"
MUSIC_CSV = DATA_DIR / "song.csv"
OUTPUT_DIR = DATA_DIR / "embeddings"

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 32
FIELDS = ("title", "genres", "creator", "plot")
UPLOAD_CHUNK = 500

## 데이터셋

`MOVIE_CSV` / `WEBTOON_CSV` / `MUSIC_CSV` — `title`·`genres`·`creator`·`plot`(영화/웹툰: 줄거리+mood_kor 합침, 음악: plot) 레코드로 변환 후 임베딩.

In [12]:
def txt(value) -> str:
    return "" if pd.isna(value) else str(value).strip()


def combine_plot_mood(plot: str, mood_kor: str = "") -> str:
    plot, mood = txt(plot), txt(mood_kor)
    if plot and mood:
        return f"{plot}\n분위기: {mood}"
    return plot or mood


def encode_pairs(pairs: list[tuple[str, str]], model) -> tuple[list[str], list[str], list[list[float]], int]:
    source_ids, texts = zip(*pairs)
    vectors = model.encode(
        list(texts),
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    dim = len(vectors[0])
    return list(source_ids), list(texts), [v.tolist() for v in vectors], dim


def build_column_items(
    media: str, field: str, records: list[dict], model, model_name: str
) -> list[dict]:
    pairs = [(r["source_id"], r[field]) for r in records if r[field]]
    if not pairs:
        return []
    source_ids, texts, embeddings, dim = encode_pairs(pairs, model)
    return [
        {
            "media_type": media,
            "source_id": sid,
            "field": field,
            "text": text,
            "embedding": emb,
            "model": model_name,
            "dim": dim,
        }
        for sid, text, emb in zip(source_ids, texts, embeddings)
    ]


def save_items_csv(path: Path, items: list[dict], field: str | None = None) -> None:
    rows = []
    for item in items:
        row = {
            "source_id": item["source_id"],
            "text": item["text"],
            "embedding": json.dumps(item["embedding"]),
            "model": item["model"],
            "dim": item["dim"],
        }
        if field:
            row["field"] = field
        rows.append(row)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[csv] {path.name}: {len(rows)} rows")

In [13]:
def prepare_records(df: pd.DataFrame, media: str) -> list[dict]:
    records = []
    for _, row in df.iterrows():
        if media == "movie":
            keywords, overview = txt(row["keywords"]), txt(row["overview"])
            plot = f"{keywords}. {overview}" if keywords else overview
            director, cast = txt(row["director"]), txt(row["cast"])
            creator = ", ".join(x for x in (director, cast) if x)
            records.append(
                {
                    "source_id": str(int(row["movie_id"])),
                    "title": txt(row["title"]),
                    "genres": txt(row["genres"]),
                    "creator": creator,
                    "plot": combine_plot_mood(plot, row["poster_mood_kor"]),
                }
            )
        elif media == "webtoon":
            records.append(
                {
                    "source_id": str(int(row["webtoon_id"])),
                    "title": txt(row["series_title"]),
                    "genres": txt(row["genres"]),
                    "creator": txt(row["author"]),
                    "plot": combine_plot_mood(row["description"], row["mood_kor"]),
                }
            )
        elif media == "music":
            records.append(
                {
                    "source_id": txt(row["track_id"]),
                    "title": txt(row["track_name"]),
                    "genres": txt(row["genre"]),
                    "creator": txt(row["track_artist"]),
                    "plot": txt(row["plot"]),
                }
            )
        else:
            raise ValueError(f"unknown media: {media}")
    return records


movie_df = pd.read_csv(MOVIE_CSV)
webtoon_df = pd.read_csv(WEBTOON_CSV)
music_df = pd.read_csv(MUSIC_CSV)

movie_rows = prepare_records(movie_df, "movie")
webtoon_rows = prepare_records(webtoon_df, "webtoon")
music_rows = prepare_records(music_df, "music")

DATASETS = [
    ("movie", movie_rows),
    ("webtoon", webtoon_rows),
    ("music", music_rows),
]

print(f"영화 {len(movie_rows)}건, 웹툰 {len(webtoon_rows)}건, 음악 {len(music_rows)}건")
print("영화 샘플:", movie_rows[0])
print("웹툰 샘플:", webtoon_rows[0])
print("음악 샘플:", music_rows[0])

영화 1609건, 웹툰 297건, 음악 1437건
영화 샘플: {'source_id': '1304313', 'title': '리 크로닌의 미이라', 'genres': '스릴러/범죄, 공포/호러', 'creator': '리 크로닌, 잭 레이너, 라이아 코스타, 메이 칼라마위', 'plot': 'journalist, egypt, monster, ritual, kidnapping. 한 기자의 어린 딸이 흔적도 없이 사막으로 사라진다. 그리고 8년 후, 산산이 부서진 가족은 그녀가 다시 돌아왔다는 소식에 충격을 받는다. 하지만 기쁨으로 가득해야 할 재회는 곧 살아 있는 악몽으로 변해간다.\n분위기: 어두운 시네마틱 느와르'}
웹툰 샘플: {'source_id': '393727271', 'title': '전지적 독자 시점', 'genres': '판타지', 'creator': '슬리피-C (지은이), 싱숑 (원작), UMI (각색)', 'plot': '낙원의 멸망을 딛고 비상한 키메라 드래곤. 그리고 일행 앞에 주어진 새로운 시련. 김독자는 가장 사랑하는 존재에게 죽게 될 것이다. 예언대로 됐을 뿐이야. 독자야, 나는 세상 누구보다 너를 사랑한단다. 어쩌면 나 자신보다 더. 지금부터 모든 걸 ‘다시 읽을’ 거란다.\n분위기: 역동적인 액션'}
음악 샘플: {'source_id': '2plbrEY59IikOBgBGLjaoe', 'title': 'Die With A Smile', 'genres': 'pop,global', 'creator': 'Lady Gaga, Bruno Mars', 'plot': '"Die With A Smile"는 중간 정도의 에너지를 지닌 곡으로, 감정적으로 깊이 있는 분위기를 전달합니다. 비트가 빠르고 댄스하기 좋은 요소를 갖추고 있지만, 단조의 조화가 곡에 약간의 쓸쓸함을 더해줍니다. 전체적으로는 경쾌하면서도 감성적인 느낌이 어우러져, 듣는 이에게 복잡한 감정을 불러일으키는 매력이 있습니다.'}


### 모델 로드

In [14]:
print(f"모델 로딩: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

모델 로딩: paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11703.61it/s]


### 컬럼별 임베딩

`build_column_items` — `title` · `genres` · `creator` · `plot` 각각 따로 임베딩.

In [15]:
column_items: dict[tuple[str, str], list[dict]] = {}

for media, records in DATASETS: 
    for field in FIELDS:
        items = build_column_items(media, field, records, model, MODEL_NAME)
        column_items[(media, field)] = items
        print(f"[column] {media} {field}: {len(items)}건")

Batches: 100%|██████████| 51/51 [00:02<00:00, 25.24it/s]


[column] movie title: 1609건


Batches: 100%|██████████| 51/51 [00:01<00:00, 25.98it/s]


[column] movie genres: 1609건


Batches: 100%|██████████| 51/51 [00:05<00:00, 10.17it/s]


...

[column] music genres: 1437건


Batches: 100%|██████████| 45/45 [00:01<00:00, 26.95it/s]


[column] music creator: 1437건


Batches: 100%|██████████| 45/45 [00:15<00:00,  2.93it/s]

[column] music plot: 1437건


### CSV 저장

In [6]:
def export_embeddings_csv(column_items: dict[tuple[str, str], list[dict]]) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for (media, field), items in column_items.items():
        save_items_csv(
            OUTPUT_DIR / f"{media}_embedding_{field}.csv",
            items,
            field=field,
        )
    print(f"[csv] 저장 완료 → {OUTPUT_DIR}")

In [8]:
export_embeddings_csv(column_items)

[csv] movie_embedding_title.csv: 1609 rows
[csv] movie_embedding_genres.csv: 1609 rows
[csv] movie_embedding_creator.csv: 1609 rows
[csv] movie_embedding_plot_mood.csv: 1609 rows
[csv] webtoon_embedding_title.csv: 297 rows
[csv] webtoon_embedding_genres.csv: 297 rows
[csv] webtoon_embedding_creator.csv: 297 rows
[csv] webtoon_embedding_plot_mood.csv: 297 rows
[csv] music_embedding_title.csv: 1437 rows
[csv] music_embedding_genres.csv: 1437 rows
[csv] music_embedding_creator.csv: 1437 rows
[csv] music_embedding_plot_mood.csv: 1437 rows
[csv] 저장 완료 → C:\Users\USER\Desktop\code\3-Idiots\Project\raw_data\embeddings


## API 업로드

In [20]:
import os

import requests
from dotenv import load_dotenv

load_dotenv(ENV_FILE)
BASE_URL = os.environ["BASE_URL"].rstrip("/")
ADMIN_API_KEY = os.environ["ADMIN_API_KEY"]
API_HEADERS = {"X-API-Key": ADMIN_API_KEY}


def post_items(items: list[dict]) -> int:
    if not items:
        return 0
    url = f"{BASE_URL}/embeddings/add"
    total = 0
    for i in range(0, len(items), UPLOAD_CHUNK):
        resp = requests.post(
            url,
            json={"items": items[i : i + UPLOAD_CHUNK]},
            headers=API_HEADERS,
            timeout=180,
        )
        if not resp.ok:
            raise RuntimeError(f"{resp.status_code} column {resp.text}")
        total += resp.json()["upserted"]
    return total    


def upload_embeddings_api(column_items: dict[tuple[str, str], list[dict]]) -> None:
    for (media, field), items in column_items.items():
        print(f"[column] {media} {field}: {post_items(items)}")

In [21]:
upload_embeddings_api(column_items)

[column] movie title: 1609
[column] movie genres: 1609
[column] movie creator: 1609
[column] movie plot: 1609
[column] webtoon title: 297
[column] webtoon genres: 297
[column] webtoon creator: 297
[column] webtoon plot: 297
[column] music title: 1437
[column] music genres: 1437
[column] music creator: 1437
[column] music plot: 1437
